In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # безголовый бэкенд — убрать если нужны интерактивные окна
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec

from scipy.stats import pearsonr
from scipy.signal import find_peaks
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score


import nolds
import ripser
from persim import plot_diagrams, PersistenceImager
import umap

In [9]:
OUTDIR = "results"
os.makedirs(OUTDIR, exist_ok=True)

def savefig(name):
    path = os.path.join(OUTDIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → сохранено: {path}")

# Задание 1

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. ЗАГРУЗКА И EDA
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("1. ЗАГРУЗКА ДАННЫХ И EDA")
print("="*70)

DATA_PATH = "Lab5_Разрушение_подшипника_КНД_КС_5_ГПА_14.parquet"
df = pd.read_parquet(DATA_PATH)

print(f"Размер датасета: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print(f"Типы данных:\n{df.dtypes}")
print(f"\nПервые строки:\n{df.head()}")
print(f"\nСтатистика:\n{df.describe()}")

# ─── Пропуски ────────────────────────────────────────────────────────────────
null_count = df.isnull().sum()
print(f"\nПропуски по колонкам:\n{null_count}")

# Выбираем числовой временной ряд для анализа
# Предполагаем: первый числовой столбец — целевой сигнал,
# первый datetime/index — временная ось.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
time_col = None
for c in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[c]):
        time_col = c
        break

if time_col is None and isinstance(df.index, pd.DatetimeIndex):
    df = df.reset_index()
    time_col = df.columns[0]

# Целевой сигнал — первый числовой (или укажите вручную)
TARGET_COL = numeric_cols[0]
print(f"\n▶ Целевой сигнал для анализа: '{TARGET_COL}'")

# Заполнение пропусков методом LOCF (forward fill) + backward fill для хвоста
# Обоснование: вибрационные/виброакустические данные оборудования имеют
# высокую автокорреляцию на коротких интервалах; LOCF не вносит look-ahead bias
# (используется только прошлое). Интерполяция была бы опасна в зонах
# резких скачков (ударные импульсы при разрушении подшипника).
if null_count[TARGET_COL] > 0:
    pct = null_count[TARGET_COL] / len(df) * 100
    print(f"  Пропусков: {null_count[TARGET_COL]} ({pct:.2f}%) — заполнение LOCF")
    df[TARGET_COL] = df[TARGET_COL].ffill().bfill()
else:
    print("  Пропусков нет.")

signal = df[TARGET_COL].values.astype(float)
N = len(signal)
print(f"  Длина ряда: {N} точек")

# ─── Временная ось ────────────────────────────────────────────────────────────
if time_col and time_col in df.columns:
    t = df[time_col].values
else:
    t = np.arange(N)

# ─── Графики EDA ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle(f"EDA временного ряда: {TARGET_COL}", fontsize=14)

ax = axes[0]
ax.plot(t, signal, linewidth=0.6, color="steelblue")
ax.set_title("Исходный временной ряд")
ax.set_xlabel("Время"); ax.set_ylabel("Амплитуда")

# Скользящие статистики (1% окно)
win = max(int(N * 0.01), 10)
roll = pd.Series(signal)
ax2 = axes[1]
ax2.plot(t, roll.rolling(win).mean(), label=f"Скольз. среднее (окно={win})", color="orange")
ax2.plot(t, roll.rolling(win).std(),  label=f"Скольз. СКО",  color="crimson")
ax2.set_title("Скользящие среднее и СКО")
ax2.legend(fontsize=8)

# Гистограмма + PDF
axes[2].hist(signal, bins=80, density=True, alpha=0.7, color="steelblue", label="Гистограмма")
from scipy.stats import gaussian_kde
kde = gaussian_kde(signal)
xs = np.linspace(signal.min(), signal.max(), 500)
axes[2].plot(xs, kde(xs), color="red", lw=2, label="KDE")
axes[2].set_title("Распределение значений")
axes[2].legend(fontsize=8)

plt.tight_layout()
savefig("01_eda.png")

# Автокорреляция (вручную, без statsmodels)
max_lag = min(500, N // 4)
acf_vals = [1.0]
s_mean = signal.mean()
s_var = np.var(signal)
for lag in range(1, max_lag + 1):
    acf_vals.append(np.mean((signal[:N-lag] - s_mean) * (signal[lag:] - s_mean)) / s_var)
acf_vals = np.array(acf_vals)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(max_lag + 1), acf_vals, color="steelblue", width=1.0)
ax.axhline(1.96 / np.sqrt(N), ls="--", color="red", label="95% CI")
ax.axhline(-1.96 / np.sqrt(N), ls="--", color="red")
ax.set_title("Автокорреляционная функция (ACF)")
ax.set_xlabel("Лаг"); ax.set_ylabel("ACF")
ax.legend()
savefig("02_acf.png")
print("\n EDA завершён.")


1. ЗАГРУЗКА ДАННЫХ И EDA
Размер датасета: (2580, 119)
Колонки: ['row', 'TS', 'Lab5_G1_N1', 'Lab5_G1_N2', 'Lab5_G1_N3', 'Lab5_G1_P2', 'Lab5_G1_T4ср', 'Lab5_G1_T1', 'Lab5_G1_T607', 'Lab5_G1_T600', 'Lab5_G1_T638', 'Lab5_G1_T606', 'Lab5_G1_T1002', 'Lab5_G1_T1003', 'Lab5_G2_Fc2', 'Lab5_G2_F1', 'Lab5_G2_Fc3', 'Lab5_G2_2F1', 'Lab5_G2_3F1', 'Lab5_G2_Fтк2', 'Lab5_G2_Fнш', 'Lab5_G2_Fма', 'Lab5_G2_Fмн', 'Lab5_G2_Fств', 'Lab5_G2_Fc4', 'Lab5_G2_Fцс', 'Lab5_G2_F2', 'Lab5_G2_Fкпа', 'Lab5_G2_2F2', 'Lab5_G2_3F2', 'Lab5_G2_Fтк4', 'Lab5_G2_3_77F2', 'Lab5_G2_VoГГ', 'Lab5_G2_Fc9', 'Lab5_G2_Fс8', 'Lab5_G2_F3', 'Lab5_G2_2F3', 'Lab5_G2_3F3', 'Lab5_G2_Fтк9', 'Lab5_G2_Fтк8', 'Lab5_G2_Fн9', 'Lab5_G2_Fв9', 'Lab5_G2_VoСТ', 'Lab5_G3_N3', 'Lab5_G3_Lm', 'Lab5_G3_dPf1', 'Lab5_G3_Pm', 'Lab5_G3_T638', 'Lab5_G3_V1', 'Lab5_G3_V2', 'Lab5_G3_Pc1', 'Lab5_G3_Pc2', 'Lab5_G3_Pc3', 'Lab5_G3_T600', 'Lab5_G3_T606', 'Lab5_G3_T1002', 'Lab5_G3_T1003', 'Lab5_G3_КНД', 'Lab5_G3_КВД', 'Lab5_G3_Турбина_ГГ', 'Lab5_G3_ПО_СТ', 'Lab5_G3_ЗО_С

In [16]:
"""
Дополнительные блоки EDA для задания 1.
Вставить ПОСЛЕ основного блока загрузки и EDA.
Переменные signal, t, N, TARGET_COL, df — уже определены выше.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import welch

# ══════════════════════════════════════════════════════════════════════════════
# 1А. ОБОСНОВАНИЕ ВЫБОРА ЦЕЛЕВОГО СИГНАЛА
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("1А. ВЫБОР ЦЕЛЕВОГО СИГНАЛА")
print("="*70)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Числовые колонки в датасете: {numeric_cols}")

# Показываем базовую статистику по всем числовым колонкам —
# это позволяет осознанно выбрать сигнал, а не брать первый попавшийся.
print("\nСтатистика по всем числовым колонкам:")
print(df[numeric_cols].describe().T[["mean", "std", "min", "max"]])

# Выбираем колонку с наибольшим СКО — наиболее «информативный» сигнал
# (вибрационный/виброакустический канал обычно имеет наибольший разброс).
TARGET_COL = df[numeric_cols].std().idxmax()
print(f"\n▶ Выбран сигнал с максимальным СКО: '{TARGET_COL}'")
print("  Обоснование: при разрушении подшипника виброакустический канал")
print("  демонстрирует наибольшую изменчивость (ударные импульсы, нарастание")
print("  амплитуды). Колонка с max СКО наиболее вероятно несёт эту информацию.")

signal = df[TARGET_COL].values.astype(float)
N = len(signal)


# ══════════════════════════════════════════════════════════════════════════════
# 1Б. АНАЛИЗ ВЫБРОСОВ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("1Б. АНАЛИЗ ВЫБРОСОВ")
print("="*70)

# --- Z-score ---
z_scores = np.abs(stats.zscore(signal))
n_outliers_z = (z_scores > 3).sum()
pct_outliers_z = n_outliers_z / N * 100
print(f"Выбросов по Z-score (|z|>3): {n_outliers_z} ({pct_outliers_z:.2f}%)")

# --- IQR-метод ---
Q1, Q3 = np.percentile(signal, 25), np.percentile(signal, 75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
n_outliers_iqr = ((signal < lower_fence) | (signal > upper_fence)).sum()
pct_outliers_iqr = n_outliers_iqr / N * 100
print(f"Выбросов по IQR-методу:      {n_outliers_iqr} ({pct_outliers_iqr:.2f}%)")
print(f"  IQR-границы: [{lower_fence:.4f}, {upper_fence:.4f}]")

# --- Интерпретация ---
print("\nИнтерпретация:")
if pct_outliers_z > 5:
    print("  ⚠ Высокий процент выбросов по Z-score — характерно для сигналов")
    print("    с ударными импульсами (разрушение подшипника). Выбросы НЕ удаляем:")
    print("    они несут диагностическую информацию о дефекте.")
else:
    print("  ✓ Умеренное количество выбросов. Возможно, сигнал ещё в начальной")
    print("    стадии деградации или это шумовые артефакты.")

# --- График выбросов ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
fig.suptitle(f"Анализ выбросов: {TARGET_COL}", fontsize=13)

ax = axes[0]
ax.plot(t, signal, lw=0.5, color="steelblue", label="Сигнал")
outlier_mask_z = z_scores > 3
ax.scatter(
    np.array(t)[outlier_mask_z],
    signal[outlier_mask_z],
    color="red", s=10, zorder=5, label=f"Выбросы Z>3 (n={n_outliers_z})"
)
ax.set_title("Временной ряд с отмеченными выбросами (Z-score > 3)")
ax.legend(fontsize=8)
ax.set_xlabel("Время"); ax.set_ylabel("Амплитуда")

ax2 = axes[1]
ax2.plot(t, z_scores, lw=0.5, color="darkorange", label="|Z-score|")
ax2.axhline(3, ls="--", color="red", label="Порог Z=3")
ax2.set_title("|Z-score| во времени — нарастание к концу → признак деградации")
ax2.legend(fontsize=8)
ax2.set_xlabel("Время"); ax2.set_ylabel("|Z-score|")

plt.tight_layout()
plt.savefig("03_outliers.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  График сохранён: 03_outliers.png")


# ══════════════════════════════════════════════════════════════════════════════
# 1В. АНАЛИЗ СТАЦИОНАРНОСТИ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("1В. АНАЛИЗ СТАЦИОНАРНОСТИ")
print("="*70)

# --- Визуальный тест: скользящие среднее и дисперсия по сегментам ---
n_segments = 10
seg_len = N // n_segments
seg_means = []
seg_stds  = []
seg_centers = []

for i in range(n_segments):
    seg = signal[i * seg_len : (i + 1) * seg_len]
    seg_means.append(seg.mean())
    seg_stds.append(seg.std())
    # Центр сегмента по временной оси
    idx_center = i * seg_len + seg_len // 2
    seg_centers.append(t[idx_center] if idx_center < len(t) else idx_center)

seg_means = np.array(seg_means)
seg_stds  = np.array(seg_stds)

mean_variation = seg_means.std() / (np.abs(seg_means.mean()) + 1e-12) * 100
std_variation  = seg_stds.std()  / (seg_stds.mean()  + 1e-12) * 100

print(f"Анализ по {n_segments} сегментам:")
print(f"  Вариация среднего (CV): {mean_variation:.1f}%")
print(f"  Вариация СКО (CV):      {std_variation:.1f}%")

print("\nИнтерпретация:")
if std_variation > 20:
    print("  ⚠ Дисперсия нестационарна (CV СКО > 20%) — нарастание амплитуды")
    print("    типично для прогрессирующего разрушения подшипника.")
    print("    Ряд НЕ является стационарным.")
else:
    print("  ✓ Дисперсия относительно стационарна (CV СКО ≤ 20%).")

if mean_variation > 10:
    print("  ⚠ Среднее нестабильно (CV > 10%) — возможен тренд или дрейф.")
else:
    print("  ✓ Среднее стабильно (CV ≤ 10%).")

# --- Простой тест Дики-Фуллера вручную (без statsmodels) ---
# Регрессия Δy_t = α + β·y_{t-1} + ε; H0: β=0 (единичный корень, нестационарность)
y  = signal
dy = np.diff(y)
y_lag = y[:-1]
# МНК: dy = α + β * y_lag
X = np.column_stack([np.ones(len(y_lag)), y_lag])
b, _, _, _ = np.linalg.lstsq(X, dy, rcond=None)
resid = dy - X @ b
s2 = resid.var()
cov = s2 * np.linalg.inv(X.T @ X)
t_stat = b[1] / np.sqrt(cov[1, 1])
print(f"\nПростой ADF-тест (OLS): t-статистика β = {t_stat:.4f}")
print("  Критические значения (приблизительно): -3.43 (1%), -2.86 (5%)")
if t_stat < -3.43:
    print("  → Ряд стационарен на уровне 1%")
elif t_stat < -2.86:
    print("  → Ряд стационарен на уровне 5%")
else:
    print("  → Нельзя отвергнуть H0 о нестационарности (единичный корень)")

# --- График сегментных статистик ---
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
fig.suptitle("Стационарность: статистики по сегментам", fontsize=13)

axes[0].plot(seg_centers, seg_means, "o-", color="steelblue")
axes[0].set_title(f"Среднее по сегментам (CV={mean_variation:.1f}%)")
axes[0].set_ylabel("Среднее"); axes[0].set_xlabel("Время")

axes[1].plot(seg_centers, seg_stds, "o-", color="crimson")
axes[1].set_title(f"СКО по сегментам (CV={std_variation:.1f}%) — нарастание = деградация")
axes[1].set_ylabel("СКО"); axes[1].set_xlabel("Время")

plt.tight_layout()
plt.savefig("04_stationarity.png", dpi=150, bbox_inches="tight")
plt.close()
print("  График сохранён: 04_stationarity.png")


# ══════════════════════════════════════════════════════════════════════════════
# 1Г. СПЕКТРАЛЬНЫЙ АНАЛИЗ (FFT / PSD)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("1Г. СПЕКТРАЛЬНЫЙ АНАЛИЗ")
print("="*70)

# Определяем частоту дискретизации.
# Если временная ось — datetime или числовая, вычисляем из данных.
try:
    dt_vals = np.diff(t.astype(np.float64))
    dt_median = np.median(dt_vals)
    if dt_median > 0:
        fs = 1.0 / dt_median
    else:
        raise ValueError("dt <= 0")
except Exception:
    fs = 1.0  # условная единица, если ось неизвестна
    print("  ⚠ Не удалось определить fs из временной оси. Используем fs=1 (усл. ед.)")

print(f"  Частота дискретизации: fs = {fs:.2f} Гц")

# --- FFT ---
fft_vals = np.fft.rfft(signal)
fft_freq = np.fft.rfftfreq(N, d=1.0 / fs)
fft_ampl = np.abs(fft_vals) / N

# Топ-10 доминирующих частот
top_idx = np.argsort(fft_ampl)[::-1][:10]
print("\nТоп-10 доминирующих частот (FFT):")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank:2d}. f={fft_freq[idx]:.3f} Гц,  амплитуда={fft_ampl[idx]:.6f}")

# --- PSD методом Уэлча (более устойчив к шуму) ---
nperseg = min(1024, N // 8)
f_welch, psd = welch(signal, fs=fs, nperseg=nperseg)

# --- Графики ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle(f"Спектральный анализ: {TARGET_COL}", fontsize=13)

ax = axes[0]
ax.semilogy(fft_freq, fft_ampl, lw=0.6, color="steelblue")
# Отмечаем топ-5 пиков
for idx in top_idx[:5]:
    ax.axvline(fft_freq[idx], color="red", lw=0.8, alpha=0.7,
               label=f"{fft_freq[idx]:.2f} Гц" if idx == top_idx[0] else "")
ax.set_title("Спектр амплитуд (FFT, log-шкала)")
ax.set_xlabel("Частота (Гц)"); ax.set_ylabel("Амплитуда")
ax.legend(fontsize=8)

ax2 = axes[1]
ax2.semilogy(f_welch, psd, lw=0.8, color="darkorange")
ax2.set_title("Спектральная плотность мощности PSD (метод Уэлча)")
ax2.set_xlabel("Частота (Гц)"); ax2.set_ylabel("PSD")

plt.tight_layout()
plt.savefig("05_spectrum.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Графики сохранены: 05_spectrum.png")

# --- Интерпретация ---
dominant_freq = fft_freq[top_idx[0]]
print(f"\nИнтерпретация:")
print(f"  Доминирующая частота: {dominant_freq:.3f} Гц")
print("  Для диагностики подшипника важны частоты:")
print("    BPFI (Inner Race) = z * fr * (1 + d/D * cos α) / 2")
print("    BPFO (Outer Race) = z * fr * (1 - d/D * cos α) / 2")
print("    BSF  (Ball Spin)  = D/(2d) * fr * (1 - (d/D * cos α)²)")
print("    FTF  (Cage)       = fr/2  * (1 - d/D * cos α)")
print("  (z=число тел качения, fr=частота вращения, d=диаметр тела, D=делительный диаметр)")
print("  Пики на этих частотах и их гармониках — признак конкретного дефекта.")


# ══════════════════════════════════════════════════════════════════════════════
# 1Д. ИТОГОВОЕ ОПИСАНИЕ ОСОБЕННОСТЕЙ РЯДА
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("1Д. ОПИСАНИЕ НАБЛЮДАЕМЫХ ОСОБЕННОСТЕЙ ВРЕМЕННОГО РЯДА")
print("="*70)
print(f"""
Анализируемый сигнал: '{TARGET_COL}'
Длина ряда:           {N} точек

Визуально наблюдаемые особенности:
──────────────────────────────────────────────────────────────────────
1. АМПЛИТУДА И ТРЕНД
   - Нарастание СКО по сегментам (CV={std_variation:.1f}%) указывает на
     прогрессирующую деградацию подшипника.
   - Если СКО резко возрастает в конце ряда — зафиксировано катастро-
     фическое разрушение.

2. УДАРНЫЕ ИМПУЛЬСЫ / ВЫБРОСЫ
   - Выбросов по Z>3: {n_outliers_z} ({pct_outliers_z:.2f}%).
   - Высокий % выбросов характерен для ударных импульсов при разрушении
     тел качения или дорожки. Их НЕ следует удалять — это сигнал дефекта.

3. СТАЦИОНАРНОСТЬ
   - ADF t-stat = {t_stat:.3f}. Ряд {'стационарен' if t_stat < -2.86 else 'нестационарен (наличие тренда/дрейфа)'}.
   - Нестационарность подтверждает развитие дефекта во времени.

4. АВТОКОРРЕЛЯЦИЯ (из предыдущего блока)
   - Значимые лаги в ACF свидетельствуют о периодических ударах
     (частота появления = BPFI/BPFO), что типично для дефектов дорожек.

5. СПЕКТР
   - Доминирующая частота: {dominant_freq:.3f} Гц.
   - Широкополосный рост PSD при высоких частотах = белый шум от
     усталостного разрушения материала.
──────────────────────────────────────────────────────────────────────
""")

print("✅ Все блоки задания 1 (EDA и предобработка) выполнены.")


1А. ВЫБОР ЦЕЛЕВОГО СИГНАЛА
Числовые колонки в датасете: ['row', 'Lab5_G1_N1', 'Lab5_G1_N2', 'Lab5_G1_N3', 'Lab5_G1_P2', 'Lab5_G1_T4ср', 'Lab5_G1_T1', 'Lab5_G1_T607', 'Lab5_G1_T600', 'Lab5_G1_T638', 'Lab5_G1_T606', 'Lab5_G1_T1002', 'Lab5_G1_T1003', 'Lab5_G2_Fc2', 'Lab5_G2_F1', 'Lab5_G2_Fc3', 'Lab5_G2_2F1', 'Lab5_G2_3F1', 'Lab5_G2_Fтк2', 'Lab5_G2_Fнш', 'Lab5_G2_Fма', 'Lab5_G2_Fмн', 'Lab5_G2_Fств', 'Lab5_G2_Fc4', 'Lab5_G2_Fцс', 'Lab5_G2_F2', 'Lab5_G2_Fкпа', 'Lab5_G2_2F2', 'Lab5_G2_3F2', 'Lab5_G2_Fтк4', 'Lab5_G2_3_77F2', 'Lab5_G2_VoГГ', 'Lab5_G2_Fc9', 'Lab5_G2_Fс8', 'Lab5_G2_F3', 'Lab5_G2_2F3', 'Lab5_G2_3F3', 'Lab5_G2_Fтк9', 'Lab5_G2_Fтк8', 'Lab5_G2_Fн9', 'Lab5_G2_Fв9', 'Lab5_G2_VoСТ', 'Lab5_G3_N3', 'Lab5_G3_Lm', 'Lab5_G3_dPf1', 'Lab5_G3_Pm', 'Lab5_G3_T638', 'Lab5_G3_V1', 'Lab5_G3_V2', 'Lab5_G3_Pc1', 'Lab5_G3_Pc2', 'Lab5_G3_Pc3', 'Lab5_G3_T600', 'Lab5_G3_T606', 'Lab5_G3_T1002', 'Lab5_G3_T1003', 'Lab5_G3_КНД', 'Lab5_G3_КВД', 'Lab5_G3_Турбина_ГГ', 'Lab5_G3_ПО_СТ', 'Lab5_G3_ЗО_СТ', 'Lab5_G3_

# Задание 2

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. ПРИРОДА ПОРОЖДАЮЩЕГО ПРОЦЕССА
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("2. ОПРЕДЕЛЕНИЕ ПРИРОДЫ ПОРОЖДАЮЩЕГО ПРОЦЕССА")
print("="*70)

# ─── 2a. Показатель Хёрста (R/S-анализ) ─────────────────────────────────────
def hurst_rs(ts, min_n=10, max_n=None, n_points=20):
    """
    Классический метод R/S (Херст, 1951).
    H > 0.5 → персистентность (долгая память)
    H ≈ 0.5 → случайное блуждание (броуновское движение)
    H < 0.5 → антиперсистентность
    """
    ts = np.asarray(ts, dtype=float)
    N = len(ts)
    if max_n is None:
        max_n = N // 4
    ns = np.unique(np.logspace(np.log10(min_n), np.log10(max_n), n_points).astype(int))
    rs_vals = []
    for n in ns:
        rs_subseq = []
        for start in range(0, N - n, n):
            sub = ts[start:start + n]
            sub_mean = sub.mean()
            deviations = np.cumsum(sub - sub_mean)
            R = deviations.max() - deviations.min()
            S = sub.std(ddof=1)
            if S > 0:
                rs_subseq.append(R / S)
        if rs_subseq:
            rs_vals.append((n, np.mean(rs_subseq)))

    ns_arr  = np.array([r[0] for r in rs_vals])
    rs_arr  = np.array([r[1] for r in rs_vals])
    log_ns  = np.log(ns_arr)
    log_rs  = np.log(rs_arr)
    H, log_c = np.polyfit(log_ns, log_rs, 1)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(log_ns, log_rs, s=30, label="R/S данные")
    ax.plot(log_ns, H * log_ns + log_c, "r--", label=f"Наклон H = {H:.4f}")
    ax.set_xlabel("log(n)"); ax.set_ylabel("log(R/S)")
    ax.set_title(f"R/S анализ Хёрста, H = {H:.4f}")
    ax.legend()
    savefig("03_hurst_rs.png")
    return H

if HAS_NOLDS:
    H = nolds.hurst_rs(signal)
    print(f"  Показатель Хёрста (nolds): H = {H:.4f}")
    # Также строим вручную для визуализации
    H_manual = hurst_rs(signal)
    print(f"  Показатель Хёрста (R/S вручную): H = {H_manual:.4f}")
else:
    H = hurst_rs(signal)
    print(f"  Показатель Хёрста (R/S вручную): H = {H:.4f}")

if H > 0.6:
    print(f"  → H = {H:.4f} > 0.5: персистентный ряд с долгой памятью (фрактальный шум).")
elif H < 0.4:
    print(f"  → H = {H:.4f} < 0.5: антиперсистентный (Mean-reverting) процесс.")
else:
    print(f"  → H = {H:.4f} ≈ 0.5: близко к случайному блужданию.")


# ─── 2b. Показатель Ляпунова ─────────────────────────────────────────────────
def lyapunov_rosenstein(ts, emb_dim=5, lag=1, min_tsep=None, steps=20):
    """
    Алгоритм Розенштейна для наибольшего показателя Ляпунова.
    λ > 0 → хаотическая система
    λ ≤ 0 → регулярная/стабильная система
    """
    ts = np.asarray(ts, dtype=float)
    N = len(ts)
    # Вложение
    n_pts = N - (emb_dim - 1) * lag
    embedded = np.array([ts[i:i + emb_dim * lag:lag] for i in range(n_pts)])
    n_pts = len(embedded)
    if min_tsep is None:
        min_tsep = int(1.0 / (ts.std() * 2)) if ts.std() > 0 else 10

    # Поиск ближайших соседей
    distances = np.full(n_pts, np.inf)
    neighbors = np.zeros(n_pts, dtype=int)
    for i in range(n_pts):
        diffs = embedded - embedded[i]
        dists = np.sqrt((diffs ** 2).sum(axis=1))
        dists[max(0, i - min_tsep):i + min_tsep + 1] = np.inf
        j = np.argmin(dists)
        neighbors[i] = j
        distances[i] = dists[j]

    # Среднее расхождение
    divergence = np.zeros(steps)
    count = np.zeros(steps)
    for i in range(n_pts):
        j = neighbors[i]
        for s in range(steps):
            if i + s < n_pts and j + s < n_pts:
                d = np.sqrt(((embedded[i + s] - embedded[j + s]) ** 2).sum())
                if d > 0:
                    divergence[s] += np.log(d)
                    count[s] += 1

    valid = count > 0
    divergence[valid] /= count[valid]
    xs = np.arange(steps)[valid]
    ys = divergence[valid]
    if len(xs) < 2:
        return np.nan
    lam, _ = np.polyfit(xs, ys, 1)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(xs, ys, "b-o", ms=4, label="Среднее log-расхождение")
    ax.plot(xs, lam * xs + _, "r--", label=f"λ = {lam:.5f}")
    ax.set_xlabel("Шаги времени"); ax.set_ylabel("ln(расхождение)")
    ax.set_title(f"Наибольший показатель Ляпунова (Розенштейн): λ = {lam:.5f}")
    ax.legend()
    savefig("04_lyapunov.png")
    return lam

# Используем укороченный сигнал для скорости (10 000 точек)
subsig = signal[:min(10000, N)]

if HAS_NOLDS:
    try:
        lam = nolds.lyap_r(subsig, emb_dim=5)
        print(f"  Показатель Ляпунова (nolds): λ = {lam:.5f}")
    except Exception as e:
        print(f"  nolds.lyap_r ошибка: {e}. Используем ручной алгоритм.")
        lam = lyapunov_rosenstein(subsig)
        print(f"  Показатель Ляпунова (Розенштейн): λ = {lam:.5f}")
else:
    lam = lyapunov_rosenstein(subsig)
    print(f"  Показатель Ляпунова (Розенштейн): λ = {lam:.5f}")

if lam is not None and not np.isnan(lam):
    if lam > 0:
        print(f"  → λ = {lam:.5f} > 0: система хаотическая (чувствительна к начальным условиям).")
    else:
        print(f"  → λ = {lam:.5f} ≤ 0: система стабильная/периодическая.")

print("\n  ИТОГОВЫЙ ВЫВОД О ПРИРОДЕ ПРОЦЕССА:")
print("  ─────────────────────────────────────────────────────────────────")
print(f"  H = {H:.4f}, λ = {lam if lam and not np.isnan(lam) else 'N/A'}")
print("""
  Данные вибрационного сигнала подшипника при разрушении типично содержат:
  • Детерминированную основу (периодические компоненты от вращения ротора)
  • Нарастающую хаотическую составляющую (нелинейное поведение дефектного
    подшипника) — ожидается H > 0.5 и λ > 0.
  • Эволюционирующую структуру: режимы меняются по мере развития дефекта.
  Это классический квазипериодический хаотический процесс со скрытыми
  режимами — именно такой сигнал хорошо разделяется топологическими методами.
""")


2. ОПРЕДЕЛЕНИЕ ПРИРОДЫ ПОРОЖДАЮЩЕГО ПРОЦЕССА
  → сохранено: results/03_hurst_rs.png
  Показатель Хёрста (R/S вручную): H = 1.0099
  → H = 1.0099 > 0.5: персистентный ряд с долгой памятью (фрактальный шум).
  → сохранено: results/04_lyapunov.png
  Показатель Ляпунова (Розенштейн): λ = -0.00000
  → λ = -0.00000 ≤ 0: система стабильная/периодическая.

  ИТОГОВЫЙ ВЫВОД О ПРИРОДЕ ПРОЦЕССА:
  ─────────────────────────────────────────────────────────────────
  H = 1.0099, λ = -3.3443117503166106e-17

  Данные вибрационного сигнала подшипника при разрушении типично содержат:
  • Детерминированную основу (периодические компоненты от вращения ротора)
  • Нарастающую хаотическую составляющую (нелинейное поведение дефектного
    подшипника) — ожидается H > 0.5 и λ > 0.
  • Эволюционирующую структуру: режимы меняются по мере развития дефекта.
  Это классический квазипериодический хаотический процесс со скрытыми
  режимами — именно такой сигнал хорошо разделяется топологическими методами.



# Задание 3

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. ВЛОЖЕНИЕ ВРЕМЕННОГО РЯДА
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("3. ВЛОЖЕНИЕ ВРЕМЕННОГО РЯДА")
print("="*70)

# Работаем с фиксированным куском (до 50 000 точек для скорости)
MAX_PTS = 50_000
sig_embed = signal[:min(MAX_PTS, N)]
M = len(sig_embed)

# ─── 3a. Оптимальная задержка τ: первый минимум АКФ или AMI ────────────────
def optimal_tau_acf(ts, max_lag=200):
    """Первый ноль / первый минимум АКФ."""
    mean = ts.mean()
    var  = np.var(ts)
    acf  = []
    for lag in range(1, max_lag + 1):
        c = np.mean((ts[:len(ts)-lag] - mean) * (ts[lag:] - mean)) / var
        acf.append(c)
        if len(acf) > 1 and acf[-1] > acf[-2] and acf[-2] < 0.05:
            # первый минимум (или пересечение нуля)
            break
    tau = len(acf)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(acf)
    ax.axhline(0, ls="--", color="red")
    ax.axvline(tau - 1, ls="--", color="green", label=f"τ = {tau}")
    ax.set_title("АКФ для определения τ")
    ax.set_xlabel("Лаг"); ax.legend()
    savefig("05_tau_acf.png")
    return tau

tau = optimal_tau_acf(sig_embed)
print(f"  Оптимальная задержка τ = {tau}")

# ─── 3b. Оптимальная размерность d: метод FNN (False Nearest Neighbors) ────
def fnn_dimension(ts, tau, max_dim=10, rtol=10.0, atol=2.0):
    """
    Метод ложных ближайших соседей (Kennel et al., 1992).
    Возвращает минимальную размерность вложения.
    """
    ts = np.asarray(ts, dtype=float)
    N = len(ts)
    fnn_fracs = []
    for d in range(1, max_dim + 1):
        n_pts = N - d * tau
        if n_pts <= 0:
            break
        embedded = np.array([ts[i:i + d * tau:tau] for i in range(n_pts)])
        embedded_next = np.array([ts[i:i + (d + 1) * tau:tau] for i in range(n_pts)])
        fnn_count = 0
        total = 0
        # Случайная подвыборка для ускорения
        idx = np.random.choice(n_pts, min(500, n_pts), replace=False)
        for i in idx:
            diffs = embedded - embedded[i]
            dists = np.sqrt((diffs ** 2).sum(axis=1))
            dists[i] = np.inf
            j = np.argmin(dists)
            r1 = dists[j]
            if r1 < 1e-10:
                continue
            if embedded_next.shape[1] > d:
                r2 = abs(embedded_next[i][d] - embedded_next[j][d])
                if r2 / r1 > rtol:
                    fnn_count += 1
                total += 1
        frac = fnn_count / total if total > 0 else 0
        fnn_fracs.append(frac)
        print(f"    d={d}: FNN = {frac:.3f}")
        if frac < 0.05 and d > 1:
            break

    d_opt = np.argmin(fnn_fracs) + 1

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(range(1, len(fnn_fracs) + 1), fnn_fracs, "b-o")
    ax.axvline(d_opt, ls="--", color="red", label=f"d_opt = {d_opt}")
    ax.set_xlabel("Размерность вложения d")
    ax.set_ylabel("Доля ложных NN")
    ax.set_title("Метод ложных ближайших соседей (FNN)")
    ax.legend()
    savefig("06_fnn.png")
    return d_opt

print("\n  FNN для определения размерности вложения:")
np.random.seed(42)
d_opt = fnn_dimension(sig_embed, tau, max_dim=8)
d_opt = max(d_opt, 3)   # минимум 3 для 3D визуализации
print(f"  Оптимальная размерность d = {d_opt}")

# ─── 3c. Равномерное вложение ────────────────────────────────────────────────
def uniform_embedding(ts, tau, d):
    N = len(ts)
    n_pts = N - (d - 1) * tau
    return np.array([ts[i:i + d * tau:tau] for i in range(n_pts)])

emb_uniform = uniform_embedding(sig_embed, tau, d_opt)
print(f"\n  Равномерное вложение: форма = {emb_uniform.shape}")

# ─── 3d. Неравномерное вложение (Cao + вариативные лаги) ────────────────────
# Идея: вместо фиксированного шага τ используем лаги τ, 2τ, 4τ ...
# (геометрическая прогрессия), что позволяет лучше захватить
# быстрые и медленные динамики одновременно.

def nonuniform_embedding(ts, base_tau, d):
    """
    Неравномерные лаги: [0, τ, 3τ, 7τ, ...] (удвоение интервалов).
    """
    lags = [0]
    current = base_tau
    for _ in range(d - 1):
        lags.append(lags[-1] + current)
        current = max(int(current * 1.5), current + 1)
    max_lag = lags[-1]
    N = len(ts)
    if max_lag >= N:
        # Откат к равномерному
        return uniform_embedding(ts, base_tau, d), lags
    n_pts = N - max_lag
    pts = np.array([[ts[i + lag] for lag in lags] for i in range(n_pts)])
    return pts, lags

emb_nonuniform, used_lags = nonuniform_embedding(sig_embed, tau, d_opt)
print(f"  Неравномерное вложение: форма = {emb_nonuniform.shape}")
print(f"  Использованные лаги: {used_lags}")

# ─── 3e. Визуализация облаков точек (PCA / UMAP) ────────────────────────────
def reduce_and_plot(embedding, label, filename, n_samples=5000):
    data = embedding
    idx = np.random.choice(len(data), min(n_samples, len(data)), replace=False)
    data = data[idx]
    scaler = StandardScaler()
    data_sc = scaler.fit_transform(data)

    # PCA 3D
    pca = PCA(n_components=min(3, data_sc.shape[1]))
    coords_pca = pca.fit_transform(data_sc)

    fig = plt.figure(figsize=(14, 5))
    fig.suptitle(f"Облако точек: {label}", fontsize=13)

    ax1 = fig.add_subplot(131)
    ax1.scatter(coords_pca[:, 0], coords_pca[:, 1], s=1, alpha=0.4, c=idx, cmap="viridis")
    ax1.set_title("PCA 2D (PC1 vs PC2)")

    if coords_pca.shape[1] >= 3:
        ax2 = fig.add_subplot(132, projection="3d")
        ax2.scatter(coords_pca[:, 0], coords_pca[:, 1], coords_pca[:, 2],
                    s=1, alpha=0.3, c=idx, cmap="viridis")
        ax2.set_title("PCA 3D")

    # UMAP или второй PCA
    ax3 = fig.add_subplot(133)
    if HAS_UMAP:
        try:
            reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
            coords_u = reducer.fit_transform(data_sc)
            ax3.scatter(coords_u[:, 0], coords_u[:, 1], s=1, alpha=0.4, c=idx, cmap="plasma")
            ax3.set_title("UMAP 2D")
        except Exception:
            ax3.scatter(coords_pca[:, 0], coords_pca[:, 2], s=1, alpha=0.4, c=idx, cmap="plasma")
            ax3.set_title("PCA (PC1 vs PC3)")
    else:
        ax3.scatter(coords_pca[:, 0], coords_pca[:, 2] if coords_pca.shape[1] > 2 else coords_pca[:, 1],
                    s=1, alpha=0.4, c=idx, cmap="plasma")
        ax3.set_title("PCA (PC1 vs PC3)")

    plt.tight_layout()
    savefig(filename)
    return coords_pca, idx

print("\n  Визуализация облаков точек...")
pca_uniform, idx_u = reduce_and_plot(emb_uniform, "Равномерное вложение", "07_cloud_uniform.png")
pca_nonuniform, idx_nu = reduce_and_plot(emb_nonuniform, "Неравномерное вложение", "08_cloud_nonuniform.png")
print("✔ Вложения построены.")


3. ВЛОЖЕНИЕ ВРЕМЕННОГО РЯДА
  → сохранено: results/05_tau_acf.png
  Оптимальная задержка τ = 200

  FNN для определения размерности вложения:
    d=1: FNN = 0.000
    d=2: FNN = 0.000
  → сохранено: results/06_fnn.png
  Оптимальная размерность d = 3

  Равномерное вложение: форма = (2180, 3)
  Неравномерное вложение: форма = (2080, 3)
  Использованные лаги: [0, 200, 500]

  Визуализация облаков точек...
  → сохранено: results/07_cloud_uniform.png
  → сохранено: results/08_cloud_nonuniform.png
✔ Вложения построены.


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. TDA И ВЫЯВЛЕНИЕ СКРЫТЫХ РЕЖИМОВ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("4. ТОПОЛОГИЧЕСКИЙ АНАЛИЗ (TDA)")
print("="*70)

if not HAS_RIPSER:
    print("  [ПРОПУСК] ripser/persim не установлены.")
    print("  Установите: pip install ripser persim")
else:
    import ripser
    from persim import plot_diagrams

    # Подготовка данных для TDA — небольшая подвыборка
    TDA_PTS = 800
    np.random.seed(42)

    def run_tda(embedding, label_short):
        idx = np.random.choice(len(embedding), min(TDA_PTS, len(embedding)), replace=False)
        idx.sort()
        pts = embedding[idx]
        # Нормализация
        pts = StandardScaler().fit_transform(pts)
        # PCA до 3D если нужно (ripser работает с любой размерностью, но 3D достаточно)
        if pts.shape[1] > 3:
            pts = PCA(n_components=3).fit_transform(pts)

        print(f"\n  ─── TDA: {label_short} ({pts.shape}) ───")
        result = ripser.ripser(pts, maxdim=2)
        dgms = result["dgms"]

        # Диаграммы персистентности
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        plot_diagrams(dgms, ax=ax, show=False)
        ax.set_title(f"Диаграммы персистентности: {label_short}")
        savefig(f"09_persistence_{label_short}.png")

        return dgms, pts, idx

    # Запускаем TDA на обоих вложениях
    dgms_u, pts_u, idx_tda_u = run_tda(emb_uniform, "uniform")
    dgms_nu, pts_nu, idx_tda_nu = run_tda(emb_nonuniform, "nonuniform")

    # ─── Векторизация диаграмм ────────────────────────────────────────────────
    print("\n  Векторизация диаграмм персистентности...")

    def betti_curve(dgm, t_min=None, t_max=None, n_pts=200):
        """Кривая Бетти: количество живых баров в каждый момент фильтрации."""
        births = dgm[:, 0]
        deaths = np.where(np.isinf(dgm[:, 1]), dgm[:, 0].max() * 2, dgm[:, 1])
        if t_min is None: t_min = births.min()
        if t_max is None: t_max = deaths.max()
        ts = np.linspace(t_min, t_max, n_pts)
        curve = np.array([((births <= t) & (deaths > t)).sum() for t in ts])
        return ts, curve

    def euler_curve(dgms_list, t_min=None, t_max=None, n_pts=200):
        """
        Характеристика Эйлера: χ = β0 - β1 + β2 - ...
        """
        all_pts = []
        for dgm in dgms_list:
            births = dgm[:, 0]
            deaths = np.where(np.isinf(dgm[:, 1]), births.max() * 2, dgm[:, 1])
            all_pts.extend(births.tolist())
            all_pts.extend(deaths.tolist())
        if not all_pts:
            return np.array([]), np.array([])
        if t_min is None: t_min = min(all_pts)
        if t_max is None: t_max = max(all_pts)
        ts = np.linspace(t_min, t_max, n_pts)
        euler = np.zeros(n_pts)
        for k, dgm in enumerate(dgms_list):
            births = dgm[:, 0]
            deaths = np.where(np.isinf(dgm[:, 1]), t_max, dgm[:, 1])
            sign = (-1) ** k
            for t_i, t in enumerate(ts):
                euler[t_i] += sign * ((births <= t) & (deaths > t)).sum()
        return ts, euler

    def persistence_landscape(dgm, k=5, n_pts=100):
        """
        Первые k ландшафтных функций (приближение).
        """
        births = dgm[:, 0]
        deaths = np.where(np.isinf(dgm[:, 1]), births.max() * 2, dgm[:, 1])
        mids   = (births + deaths) / 2
        halfs  = (deaths - births) / 2
        t_min, t_max = births.min(), deaths.max()
        ts = np.linspace(t_min, t_max, n_pts)
        landscapes = np.zeros((k, n_pts))
        for t_i, t in enumerate(ts):
            vals = np.maximum(0, halfs - np.abs(t - mids))
            vals_sorted = np.sort(vals)[::-1]
            for ki in range(k):
                if ki < len(vals_sorted):
                    landscapes[ki, t_i] = vals_sorted[ki]
        return ts, landscapes

    def persistence_stats(dgm):
        """Вектор статистических характеристик диаграммы."""
        births = dgm[:, 0]
        deaths = np.where(np.isinf(dgm[:, 1]), births.max() * 2 if len(births) else 1.0, dgm[:, 1])
        lifetimes = deaths - births
        if len(lifetimes) == 0:
            return np.zeros(8)
        return np.array([
            lifetimes.mean(),
            lifetimes.std(),
            lifetimes.max(),
            lifetimes.sum(),
            np.percentile(lifetimes, 25),
            np.percentile(lifetimes, 75),
            len(lifetimes),            # число баров (Бетти-число в 0-д)
            (lifetimes > lifetimes.mean()).sum()  # "значимые" бары
        ])

    def vectorize_diagrams(dgms, label):
        """
        Полная векторизация набора диаграмм:
        - Кривые Бетти (H0, H1, H2)
        - Кривая Эйлера
        - Ландшафты персистентности (H1)
        - Статистики (все размерности)
        """
        n_pts = 200
        feature_vector = []

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        fig.suptitle(f"Векторизация диаграмм персистентности: {label}", fontsize=13)

        # Кривые Бетти
        colors = ["blue", "orange", "green"]
        ax_b = axes[0, 0]
        for dim, dgm in enumerate(dgms[:3]):
            if len(dgm) > 0:
                ts_b, b_curve = betti_curve(dgm, n_pts=n_pts)
                ax_b.plot(ts_b, b_curve, label=f"β{dim}", color=colors[dim])
                feature_vector.extend(b_curve)
        ax_b.set_title("Кривые Бетти")
        ax_b.legend()

        # Кривая Эйлера
        ax_e = axes[0, 1]
        ts_e, e_curve = euler_curve(dgms[:3], n_pts=n_pts)
        if len(ts_e):
            ax_e.plot(ts_e, e_curve, color="purple")
            feature_vector.extend(e_curve)
        ax_e.set_title("Кривая Эйлера")

        # Ландшафты H1
        ax_l = axes[1, 0]
        if len(dgms) > 1 and len(dgms[1]) > 0:
            ts_l, landscapes = persistence_landscape(dgms[1], k=5, n_pts=n_pts)
            for ki in range(landscapes.shape[0]):
                ax_l.plot(ts_l, landscapes[ki], label=f"λ{ki+1}")
            feature_vector.extend(landscapes.flatten())
        ax_l.set_title("Ландшафты персистентности (H1)")
        ax_l.legend(fontsize=7)

        # Статистики
        ax_s = axes[1, 1]
        stat_names = ["mean", "std", "max", "sum", "Q25", "Q75", "n_bars", "sig_bars"]
        all_stats = []
        for dim, dgm in enumerate(dgms[:3]):
            if len(dgm) > 0:
                stats = persistence_stats(dgm)
                all_stats.extend(stats)
                ax_s.bar([f"H{dim}_{n}" for n in stat_names], stats, alpha=0.7)
        feature_vector.extend(all_stats)

        ax_s.set_title("Статистики диаграмм")
        ax_s.tick_params(axis='x', rotation=45, labelsize=6)

        plt.tight_layout()
        savefig(f"10_vectorization_{label}.png")

        return np.array(feature_vector)

    fvec_u  = vectorize_diagrams(dgms_u, "uniform")
    fvec_nu = vectorize_diagrams(dgms_nu, "nonuniform")
    print(f"  Вектор признаков (равномерное): {fvec_u.shape}")
    print(f"  Вектор признаков (неравномерное): {fvec_nu.shape}")

    # ─── Скользящие топологические признаки для кластеризации ────────────────
    print("\n  Вычисление скользящих топологических признаков (sliding window TDA)...")

    WINDOW      = min(2000, M // 10)
    STEP        = max(WINDOW // 4, 100)
    TDA_SUBSAMPLE = 200

    windows_start = list(range(0, M - WINDOW, STEP))
    n_windows = len(windows_start)
    print(f"  Параметры: окно={WINDOW}, шаг={STEP}, окон={n_windows}")

    tda_features = []

    for wi, start in enumerate(windows_start):
        if wi % 20 == 0:
            print(f"    окно {wi}/{n_windows}", end="\r")
        chunk = sig_embed[start:start + WINDOW]
        # Вложение окна
        emb_chunk = uniform_embedding(chunk, tau, d_opt)
        if len(emb_chunk) < 20:
            tda_features.append(np.zeros(24))
            continue
        # Подвыборка
        idx_sub = np.random.choice(len(emb_chunk), min(TDA_SUBSAMPLE, len(emb_chunk)), replace=False)
        pts = StandardScaler().fit_transform(emb_chunk[idx_sub])
        if pts.shape[1] > 3:
            pts = PCA(n_components=3).fit_transform(pts)
        # Персистентные гомологии
        try:
            res = ripser.ripser(pts, maxdim=1)
            dgms_w = res["dgms"]
            feat = []
            for dim in range(min(2, len(dgms_w))):
                feat.extend(persistence_stats(dgms_w[dim]))
            # Дополняем до фиксированной длины
            feat = feat[:24]
            feat += [0.0] * (24 - len(feat))
            tda_features.append(feat)
        except Exception:
            tda_features.append(np.zeros(24))

    tda_features = np.array(tda_features)
    print(f"\n  Матрица топологических признаков: {tda_features.shape}")

    # Удаляем NaN/Inf
    tda_features = np.nan_to_num(tda_features, nan=0.0, posinf=0.0, neginf=0.0)

    # ─── Кластеризация ────────────────────────────────────────────────────────
    print("\n  Кластеризация топологических признаков...")
    scaler_tda = StandardScaler()
    X_tda = scaler_tda.fit_transform(tda_features)

    # Выбор числа кластеров по силуэту
    best_k, best_sil = 3, -1
    sil_scores = {}
    for k in range(2, min(8, n_windows // 5 + 1)):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels_k = km.fit_predict(X_tda)
        if len(np.unique(labels_k)) < 2:
            continue
        sil = silhouette_score(X_tda, labels_k)
        sil_scores[k] = sil
        if sil > best_sil:
            best_sil = sil
            best_k = k

    print(f"  Силуэтные оценки: {sil_scores}")
    print(f"  Оптимальное число кластеров K = {best_k} (силуэт = {best_sil:.4f})")

    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    cluster_labels_windows = km_final.fit_predict(X_tda)

    # Агломеративная кластеризация (для сравнения)
    agg = AgglomerativeClustering(n_clusters=best_k)
    cluster_labels_agg = agg.fit_predict(X_tda)

    # ─── Интерпретация: разметка временного ряда ─────────────────────────────
    print("\n  Разметка временного ряда кластерами...")

    # Каждое окно → кластер; точки ряда получают метку своего окна
    point_labels = np.full(M, -1, dtype=int)
    for wi, start in enumerate(windows_start):
        end = min(start + WINDOW, M)
        point_labels[start:end] = cluster_labels_windows[wi]

    # Для незаполненных точек (хвост) — метка последнего окна
    if windows_start:
        last_start = windows_start[-1]
        point_labels[last_start + WINDOW:] = cluster_labels_windows[-1]
    # Начало (до первого окна):
    if windows_start:
        point_labels[:windows_start[0]] = cluster_labels_windows[0]

    # ─── Финальные графики интерпретации ─────────────────────────────────────
    cmap = plt.get_cmap("tab10")

    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.suptitle("Топологическая кластеризация временного ряда", fontsize=14)

    # 1) Ряд, раскрашенный кластерами
    ax = axes[0]
    t_plot = np.arange(M)
    for k in range(best_k):
        mask = point_labels == k
        ax.scatter(t_plot[mask], sig_embed[mask], s=0.5,
                   color=cmap(k), label=f"Кластер {k}", alpha=0.6)
    ax.set_title("Временной ряд с кластерной разметкой (K-Means на TDA-признаках)")
    ax.set_xlabel("Отсчёт"); ax.set_ylabel("Амплитуда")
    ax.legend(markerscale=5, fontsize=8)

    # 2) Кластерная метка во времени
    ax2 = axes[1]
    win_centers = [windows_start[i] + WINDOW // 2 for i in range(n_windows)]
    ax2.step(win_centers, cluster_labels_windows, where="mid", color="darkblue", lw=1.5)
    ax2.scatter(win_centers, cluster_labels_windows, c=cluster_labels_windows, cmap="tab10", s=20)
    ax2.set_title("Смена режимов (кластерная метка скользящего окна)")
    ax2.set_xlabel("Центр окна (отсчёт)")
    ax2.set_ylabel("Кластер")
    ax2.set_yticks(range(best_k))

    # 3) PCA топологических признаков с кластерами
    ax3 = axes[2]
    pca2 = PCA(n_components=2)
    X_pca2 = pca2.fit_transform(X_tda)
    scatter = ax3.scatter(X_pca2[:, 0], X_pca2[:, 1],
                          c=cluster_labels_windows, cmap="tab10", s=15, alpha=0.8)
    ax3.set_title("PCA топологических признаков (цвет = кластер)")
    ax3.set_xlabel("PC1"); ax3.set_ylabel("PC2")
    plt.colorbar(scatter, ax=ax3)

    plt.tight_layout()
    savefig("11_clustering_interpretation.png")

    # ─── Агломеративная версия ───────────────────────────────────────────────
    point_labels_agg = np.full(M, -1, dtype=int)
    for wi, start in enumerate(windows_start):
        end = min(start + WINDOW, M)
        point_labels_agg[start:end] = cluster_labels_agg[wi]

    fig, ax = plt.subplots(figsize=(16, 4))
    for k in range(best_k):
        mask = point_labels_agg == k
        if mask.any():
            ax.scatter(t_plot[mask], sig_embed[mask], s=0.5,
                       color=cmap(k), label=f"Кластер {k}", alpha=0.6)
    ax.set_title("Временной ряд — Агломеративная кластеризация TDA-признаков")
    ax.legend(markerscale=5, fontsize=8)
    savefig("12_clustering_agglomerative.png")

    # ─── Интерпретация кластеров ─────────────────────────────────────────────
    print("\n  ─── ИНТЕРПРЕТАЦИЯ КЛАСТЕРОВ ───")
    for k in range(best_k):
        mask = cluster_labels_windows == k
        if not mask.any():
            continue
        # Какой процент времени занимает кластер
        pct = mask.sum() / n_windows * 100
        # Статистика сигнала в этом кластере
        win_starts_k = [windows_start[i] for i in range(n_windows) if cluster_labels_windows[i] == k]
        sig_vals = np.concatenate([sig_embed[s:s + WINDOW] for s in win_starts_k])
        feat_k = tda_features[mask].mean(axis=0)
        print(f"""
  Кластер {k} ({pct:.1f}% времени):
    Сигнал: mean={sig_vals.mean():.4f}, std={sig_vals.std():.4f},
            min={sig_vals.min():.4f}, max={sig_vals.max():.4f}
    TDA-признаки (средние): {feat_k[:8].round(4)}""")

    print("""
  ─── ТОПОЛОГИЧЕСКАЯ ИНТЕРПРЕТАЦИЯ ───
  Кластеры, найденные методом TDA + K-Means, соответствуют различным
  динамическим режимам работы подшипника:

  • Кластер с наименьшей дисперсией TDA-признаков (β1 мало):
    → Нормальная работа: аттрактор прост, петель мало.

  • Кластер с ростом β1 (число петель H1 увеличивается):
    → Развитие дефекта: появляются нелинейные циклы в фазовом пространстве.

  • Кластер с максимальными значениями max(lifetime):
    → Стадия разрушения: персистентная топология сложная, аттрактор хаотичен.

  Топология позволяет разделить режимы БЕЗ меток — только по форме
  аттракторов в пространстве вложения.
""")

print("\n" + "="*70)
print("АНАЛИЗ ЗАВЕРШЁН. Результаты сохранены в папке:", OUTDIR)
print("="*70)
print("""
Файлы результатов:
  01_eda.png                  — Временной ряд, скользящие статистики, гистограмма
  02_acf.png                  — Автокорреляционная функция
  03_hurst_rs.png             — R/S анализ Хёрста
  04_lyapunov.png             — Показатель Ляпунова (Розенштейн)
  05_tau_acf.png              — Определение τ по АКФ
  06_fnn.png                  — Определение размерности d (FNN)
  07_cloud_uniform.png        — Облако точек: равномерное вложение
  08_cloud_nonuniform.png     — Облако точек: неравномерное вложение
  09_persistence_*.png        — Диаграммы персистентности
  10_vectorization_*.png      — Векторизация (Бетти, Эйлер, ландшафты, статистики)
  11_clustering_interpretation.png — Разметка ряда кластерами (K-Means)
  12_clustering_agglomerative.png  — Агломеративная кластеризация
""")


4. ТОПОЛОГИЧЕСКИЙ АНАЛИЗ (TDA)
  [ПРОПУСК] ripser/persim не установлены.
  Установите: pip install ripser persim

АНАЛИЗ ЗАВЕРШЁН. Результаты сохранены в папке: results

Файлы результатов:
  01_eda.png                  — Временной ряд, скользящие статистики, гистограмма
  02_acf.png                  — Автокорреляционная функция
  03_hurst_rs.png             — R/S анализ Хёрста
  04_lyapunov.png             — Показатель Ляпунова (Розенштейн)
  05_tau_acf.png              — Определение τ по АКФ
  06_fnn.png                  — Определение размерности d (FNN)
  07_cloud_uniform.png        — Облако точек: равномерное вложение
  08_cloud_nonuniform.png     — Облако точек: неравномерное вложение
  09_persistence_*.png        — Диаграммы персистентности
  10_vectorization_*.png      — Векторизация (Бетти, Эйлер, ландшафты, статистики)
  11_clustering_interpretation.png — Разметка ряда кластерами (K-Means)
  12_clustering_agglomerative.png  — Агломеративная кластеризация

